In [1]:
# import required packages
from sklearn.cluster import DBSCAN
from sklearn.cluster import HDBSCAN
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import folium
import os
import yaml
from pprint import pprint
import pandas as pd
import geopandas as gpd
from scipy.spatial import ConvexHull
import numpy as np

from pyproj import Transformer

sns.set_theme(context='paper', style='darkgrid')

In [2]:
INDO_CRS = "EPSG:23867"
LL_CRS = "EPSG:4326"

def xy_to_ll(xy):
        return Transformer.from_crs(INDO_CRS, LL_CRS).transform(xy[0], xy[1])

def ll_to_xy(ll):
    return Transformer.from_crs(LL_CRS, INDO_CRS).transform(ll[0], ll[1])

In [ ]:
# construct dataset for intermediaries
dfs = []

for f in os.listdir('../AllDatesInstances/instances'):
    with open(os.path.join('instances', f), 'r') as file:
        data = yaml.safe_load(file)
        df_i = pd.DataFrame(data['intermediaries']).drop(['capacity', 'routes'], axis=1)
        dfs.append(df_i)

ints_df = pd.concat(dfs)

ints_df['int_lat'] = ints_df.location.apply(lambda x: x[0])
ints_df['int_lon'] = ints_df.location.apply(lambda x: x[1])

ints_df['int_x'], ints_df['int_y'] = ll_to_xy([ints_df.int_lat.values, ints_df.int_lon.values])

ints_df.drop('location', axis=1, inplace=True)

ints_df.rename(
    columns={'id': 'int_id'},
    inplace=True
)

ints_df.drop_duplicates(inplace=True)

ints_df.reset_index(drop=True, inplace=True)

ints_df.to_csv('ints.csv', index=False)

ints_df


,int_id,int_lat,int_lon,int_x,int_y
0,Dodi Lesmana,-0.39393,102.45772,884983.940054,-43621.049913
1,Purnomo,-0.68244,102.54698,894916.216941,-75575.896083
2,Isna,-0.70056,102.50660,890413.186074,-77579.179968
3,Agus Wibowo,-0.68910,102.46092,885321.983167,-76306.384048
4,Nurmala,-0.53178,102.40239,878808.787091,-58882.167437
5,yaya suhayat,-0.41070,102.47983,887447.924499,-45479.113640
6,Agus Yasir,-0.51696,102.39634,878135.299269,-57240.837871
7,Ngatinu,-0.52602,102.40703,879326.350243,-58244.664749
8,Samsuri,-0.35302,102.44236,883273.449125,-39090.326151
9,Riki Mandala,-0.39021,102.45724,884930.601578,-43209.101061


In [4]:
# construct master dataset for mills
dfs = []

for f in os.listdir('instances'):
    with open(os.path.join('instances', f), 'r') as file:
        data = yaml.safe_load(file)
        df_m = pd.DataFrame(data['mills'])
        dfs.append(df_m)

mills_df = pd.concat(dfs)

mills_df['mill_lat'] = mills_df.location.apply(lambda x: x[0])
mills_df['mill_lon'] = mills_df.location.apply(lambda x: x[1])

mills_df['mill_x'], mills_df['mill_y'] = ll_to_xy([mills_df.mill_lat.values, mills_df.mill_lon.values])

mills_df.drop('location', axis=1, inplace=True)

mills_df.rename(
    columns={'id': 'mill_id'},
    inplace=True
)

mills_df.drop_duplicates(inplace=True)

mills_df.reset_index(drop=True, inplace=True)

mills_df.to_csv('mills.csv', index=False)

mills_df

,mill_id,mill_lat,mill_lon,mill_x,mill_y
0,KELAYANG,-0.493400,102.062467,840926.794127,-54614.055765
1,NHR,-0.733834,102.524376,892391.975126,-81265.464763
2,PN,-0.374579,102.384353,876806.399799,-41475.058961
3,SAWIT JAYA MANDIRI LESTARI,-0.442814,102.256371,862538.334717,-49023.948365
4,SKIP,-0.682643,102.501522,889848.574101,-75594.658547
5,SRJ,-0.793145,102.596242,900398.271564,-87840.525932
6,BUMI PALMA,-0.598056,102.983333,943581.055886,-66264.324536
7,Inecda,-0.492910,102.367313,874901.160821,-54576.235330
8,AGRO SARIMAS INDONESIA,-0.516300,102.921496,936690.045505,-57201.501076
9,BSS,-0.707525,102.659475,907456.011786,-78363.654941


In [5]:
# construct master dataset for farmers
dfs = []

for f in os.listdir('instances'):
    with open(os.path.join('instances', f), 'r') as file:
        data = yaml.safe_load(file)
        farmers = data['farmers']
        df_f = pd.DataFrame(data['farmers']).drop(['id'], axis=1)
        dfs.append(df_f)

farmers_df = pd.concat(dfs)

farmers_df.rename(
    columns={'intermediary': 'int_id',
             'location': 'farmer_loc'},
    inplace=True
)

farmers_df = farmers_df.merge(ints_df, on="int_id", how="left")

farmers_df['farmer_lat'] = farmers_df['farmer_loc'].apply(lambda loc: loc[0])
farmers_df['farmer_lon'] = farmers_df['farmer_loc'].apply(lambda loc: loc[1])
farmers_df.drop(['farmer_loc'], axis=1, inplace=True)

farmers_df['farmer_x'], farmers_df['farmer_y'] = ll_to_xy([farmers_df.farmer_lat.values, farmers_df.farmer_lon.values])

farmers_df['offset_x'] = farmers_df['farmer_x'] - farmers_df['int_x']
farmers_df['offset_y'] = farmers_df['farmer_y'] - farmers_df['int_y']

farmers_df['distance'] = (farmers_df['offset_x'] ** 2 + farmers_df['offset_y'] ** 2) ** 0.5

farmers_df.to_csv('farmers.csv', index=False)

farmers_df

,int_id,quantity,int_lat,int_lon,int_x,int_y,farmer_lat,farmer_lon,farmer_x,farmer_y,offset_x,offset_y,distance
0,Agus Wibowo,1.3,-0.68910,102.46092,885321.983167,-76306.384048,-0.67154,102.450860,884201.977294,-74361.110710,-1120.005873,1945.273339,2244.660668
1,Agus Wibowo,1.4,-0.68910,102.46092,885321.983167,-76306.384048,-0.68183,102.474455,886831.346658,-75502.439763,1509.363491,803.944285,1710.118289
2,yaya suhayat,1.8,-0.41070,102.47983,887447.924499,-45479.113640,-0.36250,102.565950,897051.188543,-40145.396150,9603.264044,5333.717490,10985.045406
3,yaya suhayat,1.2,-0.41070,102.47983,887447.924499,-45479.113640,-0.37371,102.580730,898698.525692,-41387.529326,11250.601193,4091.584313,11971.511575
4,yaya suhayat,2.9,-0.41070,102.47983,887447.924499,-45479.113640,-0.36319,102.573310,897871.728393,-40222.135336,10423.803894,5256.978304,11674.395424
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3266,Riki Mandala,1.3,-0.39021,102.45724,884930.601578,-43209.101061,-0.37977,102.465640,885867.479871,-42053.424369,936.878293,1155.676692,1487.726303
3267,Riki Mandala,2.1,-0.39021,102.45724,884930.601578,-43209.101061,-0.35668,102.443050,883350.216121,-39495.631195,-1580.385457,3713.469867,4035.774603
3268,Riki Mandala,1.3,-0.39021,102.45724,884930.601578,-43209.101061,-0.33999,102.461410,885397.603587,-37648.258604,467.002009,5560.842457,5580.417521
3269,Syafrial,1.9,-0.39437,102.48981,888561.265951,-43671.267163,-0.40627,102.450080,884131.686741,-44987.130904,-4429.579209,-1315.863742,4620.894865
